![Noteable.ac.uk Banner](https://raw.githubusercontent.com/YusufNik/Noteable/refs/heads/main/Images/Noteable%20NB%20Header%20Banner.png)

# Interactive Vision Intelligence: From Classical Image Features to Deep Learning

## Legend

<div style="border:1px solid #b8daff; background:#d9edf7; color:#0c5460; padding:16px; margin:8px 0; border-radius:4px;">
    In <b>blue</b>, the <b>instructions</b> and <b>goals</b> are highlighted. This tells you what we are trying to achieve.
</div>
<div style="border:1px solid #c3e6cb; background:#d4edda; color:#155724; padding:16px; margin:8px 0; border-radius:4px;">
    In <b>green</b>, key <b>information</b> and <b>concept explanations</b> are highlighted.
</div>
<div style="border:1px solid #ffeeba; background:#fff3cd; color:#856404; padding:16px; margin:8px 0; border-radius:4px;">
    In <b>yellow</b>, <b>exercises</b> and <b>tasks</b> are highlighted for you to try yourself.
</div>
<div style="border:1px solid #f5c6cb; background:#f8d7da; color:#721c24; padding:16px; margin:8px 0; border-radius:4px;">
    In <b>red</b>, <b>error interpretation</b> and <b>debugging tips</b> are highlighted.
</div>

Welcome! This notebook explores **computer vision** through a clear learning journey.

We will treat images as data, inspect them visually, transform them, and then compare two different modelling styles:
- a **classical machine learning baseline** using engineered image features
- a **simple neural network** that learns useful visual patterns automatically

Our task will be image classification on a lightweight dataset that works well in teaching notebooks.

By the end, you will have:
- inspected real image examples
- applied visual preprocessing with **OpenCV** and **scikit-image**
- trained a classical **HOG + Logistic Regression** classifier
- trained a compact **PyTorch CNN**
- compared both models using proper evaluation metrics
- explored predictions interactively to see where each model succeeds and struggles


## 1. Setting Up Our Toolkit

<div style="border:1px solid #b8daff; background:#d9edf7; color:#0c5460; padding:16px; margin:8px 0; border-radius:4px;">
    <b>Goal:</b> Import the libraries we need for image loading, preprocessing, feature extraction, machine learning, deep learning, visualisation, and interactivity.
</div>

Click the code cell below and press the **<code>▶</code> Run** button in the toolbar, or use <code>Shift + Enter</code>. The print messages will help you follow our progress.

In [ ]:
print("Starting setup: importing vision, machine learning, and visualisation libraries...")

import warnings
import random
import copy
from pathlib import Path

warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd

import cv2
from skimage.feature import hog

import matplotlib.pyplot as plt

import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio

import ipywidgets as widgets
from IPython.display import display, clear_output

import torch
from torch import nn
from torch.utils.data import TensorDataset, DataLoader

from torchvision import datasets

from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, confusion_matrix, classification_report

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

pio.renderers.default = "iframe_connected"

print(f"PyTorch device in use: {device}")
print(f"Plotly renderer: {pio.renderers.default}")
print("Success! All libraries have been imported.")

## 2. Loading a Lightweight Vision Dataset

<div style="border:1px solid #b8daff; background:#d9edf7; color:#0c5460; padding:16px; margin:8px 0; border-radius:4px;">
    <b>Goal:</b> Load a small image dataset that is reliable for classroom use. We will try <b>FashionMNIST</b> first because it is visually rich and easy to understand. If a download is unavailable, we will fall back to <b>scikit-learn digits</b> so the notebook still runs end-to-end.
</div>

This gives us a practical and robust workflow: modern enough to feel exciting, but safe enough for teaching.

In [ ]:
print("Loading image data...")

DATA_ROOT = Path("./vision_intelligence_data")
DATA_ROOT.mkdir(parents=True, exist_ok=True)

# EDIT THIS VALUE: reduce this if you want faster experiments.
MAX_SAMPLES = 8000
IMAGE_SIZE = 28

try:
    train_raw = datasets.FashionMNIST(root=str(DATA_ROOT), train=True, download=True)
    test_raw = datasets.FashionMNIST(root=str(DATA_ROOT), train=False, download=True)
    images = torch.cat([train_raw.data, test_raw.data]).numpy().astype(np.uint8)
    labels = torch.cat([train_raw.targets, test_raw.targets]).numpy().astype(int)
    class_names = train_raw.classes
    dataset_name = "FashionMNIST (torchvision)"
    print("FashionMNIST loaded successfully.")
except Exception as err:
    print("FashionMNIST was unavailable, so a built-in fallback dataset will be used instead.")
    print(f"Dataset detail: {type(err).__name__}")
    digits = load_digits()
    raw_images = digits.images
    labels = digits.target.astype(int)
    upscaled_images = []
    for img in raw_images:
        img_255 = (img / 16.0 * 255).astype(np.uint8)
        img_28 = cv2.resize(img_255, (IMAGE_SIZE, IMAGE_SIZE), interpolation=cv2.INTER_CUBIC)
        upscaled_images.append(img_28)
    images = np.stack(upscaled_images)
    class_names = [f"Digit {i}" for i in range(len(np.unique(labels)))]
    dataset_name = "Digits (scikit-learn fallback, upscaled to 28x28)"
    print("Fallback dataset prepared successfully.")

all_indices = np.arange(len(images))
if len(images) > MAX_SAMPLES:
    sample_indices, _ = train_test_split(
        all_indices,
        train_size=MAX_SAMPLES,
        stratify=labels,
        random_state=SEED,
    )
    images = images[sample_indices]
    labels = labels[sample_indices]
    print(f"A stratified subset of {len(images)} images was selected for faster teaching-time execution.")

meta_df = pd.DataFrame(
    {
        "sample_id": np.arange(len(images)),
        "label_id": labels,
        "label_name": [class_names[i] for i in labels],
    }
)
meta_df["height"] = images.shape[1]
meta_df["width"] = images.shape[2]
meta_df["mean_intensity"] = images.reshape(len(images), -1).mean(axis=1)

print(f"Dataset source: {dataset_name}")
print(f"Image array shape: {images.shape}")
print(f"Number of classes: {len(class_names)}")
display(meta_df.head())

## 3. First Look at the Images

<div style="border:1px solid #c3e6cb; background:#d4edda; color:#155724; padding:16px; margin:8px 0; border-radius:4px;">
    <b>Why this matters:</b> Before we build any model, we should look carefully at the data. What do the images look like? Are the classes balanced? Do some classes seem visually similar? These simple checks make the later modelling decisions much easier to understand.
</div>

Run the next cell to see a class-balance plot and a small grid of example images.

In [ ]:
print("Exploring class balance and visual examples...")

class_counts = meta_df["label_name"].value_counts().rename_axis("label_name").reset_index(name="count")

fig_balance = px.bar(
    class_counts,
    x="label_name",
    y="count",
    color="label_name",
    text="count",
    title="Class balance in the vision dataset",
    template="plotly_white",
)
fig_balance.update_traces(textposition="outside")
fig_balance.update_layout(showlegend=False, xaxis_title="Class", yaxis_title="Number of images")
fig_balance.show()

fig, axes = plt.subplots(2, 5, figsize=(11, 5))
example_indices = []
for label_id in sorted(meta_df["label_id"].unique())[:10]:
    first_idx = meta_df.index[meta_df["label_id"] == label_id][0]
    example_indices.append(first_idx)

for ax, idx in zip(axes.ravel(), example_indices):
    ax.imshow(images[idx], cmap="gray")
    ax.set_title(meta_df.loc[idx, "label_name"], fontsize=10)
    ax.axis("off")

plt.suptitle("One example image from each visible class", fontsize=14)
plt.tight_layout()
plt.show()

## 4. Preprocessing Images with OpenCV and scikit-image

<div style="border:1px solid #c3e6cb; background:#d4edda; color:#155724; padding:16px; margin:8px 0; border-radius:4px;">
    <b>Core idea:</b> Images are arrays of numbers. By changing contrast, smoothing noise, or highlighting edges, we can make useful structure more visible. Classical vision pipelines often depend on this kind of feature engineering.
</div>

In this notebook, we will use preprocessing to support two goals:
- clearer visual understanding for the learner
- robust classical feature extraction using HOG (Histogram of Oriented Gradients)

HOG is a classic vision idea: instead of learning directly from raw pixels, it summarises how edge directions are distributed across small parts of the image.

In [ ]:
print("Preparing image processing helpers...")

def ensure_uint8(image):
    image = np.asarray(image)
    if image.dtype != np.uint8:
        image = np.clip(image, 0, 255).astype(np.uint8)
    return image

def preprocess_image(image):
    image = ensure_uint8(image)
    equalized = cv2.equalizeHist(image)
    blurred = cv2.GaussianBlur(equalized, (3, 3), 0)
    return equalized, blurred

def edge_image(image):
    image = ensure_uint8(image)
    return cv2.Canny(image, threshold1=60, threshold2=150)

def hog_features(image, return_visual=False):
    image = ensure_uint8(image)
    _, blurred = preprocess_image(image)
    if return_visual:
        features, hog_vis = hog(
            blurred,
            orientations=9,
            pixels_per_cell=(4, 4),
            cells_per_block=(2, 2),
            block_norm="L2-Hys",
            visualize=True,
            feature_vector=True,
        )
        return features, hog_vis
    return hog(
        blurred,
        orientations=9,
        pixels_per_cell=(4, 4),
        cells_per_block=(2, 2),
        block_norm="L2-Hys",
        visualize=False,
        feature_vector=True,
    )

def apply_safe_transform(image, rotation_degrees=0, blur_strength=0):
    transformed = ensure_uint8(image).copy()
    h, w = transformed.shape
    if rotation_degrees != 0:
        matrix = cv2.getRotationMatrix2D((w / 2, h / 2), rotation_degrees, 1.0)
        transformed = cv2.warpAffine(
            transformed,
            matrix,
            (w, h),
            flags=cv2.INTER_LINEAR,
            borderMode=cv2.BORDER_REFLECT,
        )
    if blur_strength > 0:
        kernel = 2 * int(blur_strength) + 1
        transformed = cv2.GaussianBlur(transformed, (kernel, kernel), 0)
    return transformed

print("Image preprocessing helpers are ready.")

<div style="border:1px solid #ffeeba; background:#fff3cd; color:#856404; padding:16px; margin:8px 0; border-radius:4px;">
    <b>Exercise:</b> In the next cell, change <code>EXAMPLE_INDEX</code> to inspect a different image. Then run the cell again with <code>Shift + Enter</code> and compare how the preprocessing views change.
</div>

In [ ]:
print("Selecting one image for close inspection...")

# EDIT THIS VALUE: choose any sample index from 0 to len(images) - 1
EXAMPLE_INDEX = 0

example_image = ensure_uint8(images[EXAMPLE_INDEX])
example_label = meta_df.loc[EXAMPLE_INDEX, "label_name"]
equalized_image, blurred_image = preprocess_image(example_image)
edges = edge_image(blurred_image)
_, hog_vis = hog_features(example_image, return_visual=True)

print(f"Dataset source: {dataset_name}")
print(f"Sample {EXAMPLE_INDEX} | class: {example_label}")

fig, axes = plt.subplots(1, 4, figsize=(14, 4))
axes[0].imshow(example_image, cmap="gray")
axes[0].set_title("Original image")
axes[1].imshow(equalized_image, cmap="gray")
axes[1].set_title("Equalised contrast")
axes[2].imshow(edges, cmap="gray")
axes[2].set_title("Edges with OpenCV")
axes[3].imshow(hog_vis, cmap="inferno")
axes[3].set_title("HOG visualisation")
for ax in axes:
    ax.axis("off")
plt.tight_layout()
plt.show()

## 5. A Classical Vision Baseline with Engineered Features

<div style="border:1px solid #b8daff; background:#d9edf7; color:#0c5460; padding:16px; margin:8px 0; border-radius:4px;">
    <b>Goal:</b> Build a strong classical baseline using <b>HOG features</b> and <b>Logistic Regression</b>. This is a classic computer vision workflow: transform each image into a feature vector, then train a traditional classifier.
</div>

We will use a **train / validation / test** split so that:
- the model learns on the training set
- we monitor decisions on the validation set
- and we keep the test set untouched until the end

This is good machine learning practice and makes our comparison with the neural model much more meaningful.

In [ ]:
print("Extracting HOG features, creating data splits, and training the classical baseline...")

all_indices = np.arange(len(images))

train_idx, temp_idx = train_test_split(
    all_indices,
    test_size=0.30,
    stratify=labels,
    random_state=SEED,
)

val_idx, test_idx = train_test_split(
    temp_idx,
    test_size=0.50,
    stratify=labels[temp_idx],
    random_state=SEED,
)

meta_df["split"] = "unassigned"
meta_df.loc[train_idx, "split"] = "train"
meta_df.loc[val_idx, "split"] = "validation"
meta_df.loc[test_idx, "split"] = "test"

X_hog = np.vstack([hog_features(image) for image in images]).astype(np.float32)

def evaluate_predictions(y_true, y_pred, model_name, split_name):
    precision, recall, f1, _ = precision_recall_fscore_support(
        y_true,
        y_pred,
        average="macro",
        zero_division=0,
    )
    return {
        "model": model_name,
        "split": split_name,
        "accuracy": accuracy_score(y_true, y_pred),
        "precision": precision,
        "recall": recall,
        "f1": f1,
    }

classical_model = make_pipeline(
    StandardScaler(),
    LogisticRegression(max_iter=2000, random_state=SEED)
)

classical_model.fit(X_hog[train_idx], labels[train_idx])

classical_val_pred = classical_model.predict(X_hog[val_idx])
classical_test_pred = classical_model.predict(X_hog[test_idx])

classical_val_proba = classical_model.predict_proba(X_hog[val_idx])
classical_test_proba = classical_model.predict_proba(X_hog[test_idx])

all_classical_pred = classical_model.predict(X_hog)
all_classical_proba = classical_model.predict_proba(X_hog)

classical_val_metrics = evaluate_predictions(
    labels[val_idx],
    classical_val_pred,
    "HOG + Logistic Regression",
    "validation",
)

classical_test_metrics = evaluate_predictions(
    labels[test_idx],
    classical_test_pred,
    "HOG + Logistic Regression",
    "test",
)

classical_cm = confusion_matrix(labels[test_idx], classical_test_pred)

meta_df["classical_pred_id"] = all_classical_pred
meta_df["classical_pred"] = [class_names[i] for i in all_classical_pred]
meta_df["classical_confidence"] = all_classical_proba.max(axis=1)

print(f"Train size: {len(train_idx)} | Validation size: {len(val_idx)} | Test size: {len(test_idx)}")
print("Classical model trained successfully.")
display(pd.DataFrame([classical_val_metrics, classical_test_metrics]).round(3))
print("Classical test-set classification report:")
print(classification_report(labels[test_idx], classical_test_pred, target_names=class_names, zero_division=0))

In [ ]:
print("Visualising classical model results and a few mistakes...")

classical_cm_df = pd.DataFrame(
    classical_cm,
    index=[f"True: {name}" for name in class_names],
    columns=[f"Pred: {name}" for name in class_names],
)

fig_cm = px.imshow(
    classical_cm_df,
    text_auto=True,
    color_continuous_scale="Blues",
    aspect="auto",
    title="Classical baseline confusion matrix (test set)",
)
fig_cm.update_layout(coloraxis_showscale=False, template="plotly_white")
fig_cm.show()

classical_wrong = test_idx[classical_test_pred != labels[test_idx]]
n_show = min(6, len(classical_wrong))
if n_show > 0:
    fig, axes = plt.subplots(1, n_show, figsize=(2.6 * n_show, 3))
    if n_show == 1:
        axes = [axes]
    for ax, idx in zip(axes, classical_wrong[:n_show]):
        ax.imshow(images[idx], cmap="gray")
        ax.set_title(
            f"True: {meta_df.loc[idx, 'label_name']}\nPred: {meta_df.loc[idx, 'classical_pred']}",
            fontsize=9,
        )
        ax.axis("off")
    plt.suptitle("A few test images the classical model misclassified", fontsize=13)
    plt.tight_layout()
    plt.show()
else:
    print("No classical test-set errors to display for this run.")

## 6. A Simple Neural Vision Model with PyTorch

<div style="border:1px solid #b8daff; background:#d9edf7; color:#0c5460; padding:16px; margin:8px 0; border-radius:4px;">
    <b>Goal:</b> Train a compact convolutional neural network (CNN) directly on the image pixels. Instead of relying on hand-crafted HOG features, the CNN will learn useful visual patterns from data.
</div>

This is one of the main ideas in modern computer vision: neural networks can learn hierarchical features automatically.

We will keep the architecture intentionally lightweight so it remains practical on CPU-only infrastructure.

<div style="border:1px solid #ffeeba; background:#fff3cd; color:#856404; padding:16px; margin:8px 0; border-radius:4px;">
    <b>Exercise:</b> In the next cell, try changing <code>EPOCHS</code> or <code>BATCH_SIZE</code>. This is a safe and useful way to explore how neural training behaviour changes.
</div>

In [ ]:
print("Preparing image tensors and training a neural CNN...")

image_tensors = torch.tensor(images / 255.0, dtype=torch.float32).unsqueeze(1)
label_tensors = torch.tensor(labels, dtype=torch.long)

train_dataset = TensorDataset(image_tensors[train_idx], label_tensors[train_idx])
val_dataset = TensorDataset(image_tensors[val_idx], label_tensors[val_idx])
test_dataset = TensorDataset(image_tensors[test_idx], label_tensors[test_idx])

# EDIT THIS VALUE: batch size for neural training.
BATCH_SIZE = 64
# EDIT THIS VALUE: number of training epochs.
EPOCHS = 6
# EDIT THIS VALUE: learning rate.
LEARNING_RATE = 0.001

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)
all_loader = DataLoader(TensorDataset(image_tensors, label_tensors), batch_size=BATCH_SIZE, shuffle=False)

class SmallCNN(nn.Module):
    def __init__(self, num_classes):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(1, 16, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(16, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(32 * 7 * 7, 64),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(64, num_classes),
        )

    def forward(self, x):
        x = self.features(x)
        return self.classifier(x)

neural_model = SmallCNN(num_classes=len(class_names)).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(neural_model.parameters(), lr=LEARNING_RATE)

def evaluate_torch_model(model, loader):
    model.eval()
    losses = []
    all_labels = []
    all_preds = []
    all_probs = []
    with torch.no_grad():
        for xb, yb in loader:
            xb = xb.to(device)
            yb = yb.to(device)
            logits = model(xb)
            loss = criterion(logits, yb)
            probs = torch.softmax(logits, dim=1)
            preds = probs.argmax(dim=1)
            losses.append(loss.item())
            all_labels.extend(yb.detach().cpu().numpy().tolist())
            all_preds.extend(preds.detach().cpu().numpy().tolist())
            all_probs.extend(probs.detach().cpu().numpy().tolist())
    metrics = evaluate_predictions(np.array(all_labels), np.array(all_preds), "Neural CNN", "loader")
    return float(np.mean(losses)), metrics, np.array(all_probs), np.array(all_preds), np.array(all_labels)

history = []
best_state = copy.deepcopy(neural_model.state_dict())
best_val_f1 = -1.0

print("Training neural model...")
for epoch in range(1, EPOCHS + 1):
    neural_model.train()
    train_losses = []
    train_true = []
    train_pred = []
    for xb, yb in train_loader:
        xb = xb.to(device)
        yb = yb.to(device)
        optimizer.zero_grad()
        logits = neural_model(xb)
        loss = criterion(logits, yb)
        loss.backward()
        optimizer.step()
        train_losses.append(loss.item())
        train_true.extend(yb.detach().cpu().numpy().tolist())
        train_pred.extend(logits.argmax(dim=1).detach().cpu().numpy().tolist())

    train_metrics = evaluate_predictions(np.array(train_true), np.array(train_pred), "Neural CNN", "train")
    val_loss, val_metrics, _, _, _ = evaluate_torch_model(neural_model, val_loader)
    val_metrics["split"] = "validation"

    history.append(
        {
            "epoch": epoch,
            "train_loss": float(np.mean(train_losses)),
            "train_accuracy": train_metrics["accuracy"],
            "train_f1": train_metrics["f1"],
            "val_loss": val_loss,
            "val_accuracy": val_metrics["accuracy"],
            "val_f1": val_metrics["f1"],
        }
    )

    if val_metrics["f1"] > best_val_f1:
        best_val_f1 = val_metrics["f1"]
        best_state = copy.deepcopy(neural_model.state_dict())

    print(
        f"Epoch {epoch}/{EPOCHS} | train loss: {np.mean(train_losses):.4f} | train acc: {train_metrics['accuracy']:.3f} | val acc: {val_metrics['accuracy']:.3f} | val f1: {val_metrics['f1']:.3f}"
    )

neural_model.load_state_dict(best_state)

neural_val_loss, neural_val_metrics, _, _, _ = evaluate_torch_model(neural_model, val_loader)
neural_test_loss, neural_test_metrics, neural_test_proba, neural_test_pred, neural_test_true = evaluate_torch_model(neural_model, test_loader)
_, _, all_neural_proba, all_neural_pred, _ = evaluate_torch_model(neural_model, all_loader)

neural_val_metrics["split"] = "validation"
neural_test_metrics["split"] = "test"
neural_cm = confusion_matrix(neural_test_true, neural_test_pred)

meta_df["neural_pred_id"] = all_neural_pred
meta_df["neural_pred"] = [class_names[i] for i in all_neural_pred]
meta_df["neural_confidence"] = all_neural_proba.max(axis=1)

print("Neural model training complete.")
display(pd.DataFrame([neural_val_metrics, neural_test_metrics]).round(3))
print("Neural test-set classification report:")
print(classification_report(neural_test_true, neural_test_pred, target_names=class_names, zero_division=0))

In [ ]:
print("Visualising neural training history and neural test results...")

history_df = pd.DataFrame(history)

fig_history = go.Figure()
fig_history.add_trace(go.Scatter(x=history_df["epoch"], y=history_df["train_accuracy"], mode="lines+markers", name="Train accuracy"))
fig_history.add_trace(go.Scatter(x=history_df["epoch"], y=history_df["val_accuracy"], mode="lines+markers", name="Validation accuracy"))
fig_history.add_trace(go.Scatter(x=history_df["epoch"], y=history_df["train_f1"], mode="lines+markers", name="Train F1"))
fig_history.add_trace(go.Scatter(x=history_df["epoch"], y=history_df["val_f1"], mode="lines+markers", name="Validation F1"))
fig_history.update_layout(
    title="Neural model learning curve",
    xaxis_title="Epoch",
    yaxis_title="Score",
    yaxis=dict(range=[0, 1]),
    template="plotly_white",
)
fig_history.show()

neural_cm_df = pd.DataFrame(
    neural_cm,
    index=[f"True: {name}" for name in class_names],
    columns=[f"Pred: {name}" for name in class_names],
)

fig_neural_cm = px.imshow(
    neural_cm_df,
    text_auto=True,
    color_continuous_scale="Greens",
    aspect="auto",
    title="Neural model confusion matrix (test set)",
)
fig_neural_cm.update_layout(coloraxis_showscale=False, template="plotly_white")
fig_neural_cm.show()

## 7. Direct Comparison: Classical Features vs Learned Features

<div style="border:1px solid #c3e6cb; background:#d4edda; color:#155724; padding:16px; margin:8px 0; border-radius:4px;">
    <b>Key idea:</b> The HOG baseline and the neural CNN are solving the same task in different ways. The classical model depends on a hand-crafted representation of edges and shapes. The neural model learns its own internal features directly from the pixels.
</div>

This comparison is one of the most important lessons in modern machine learning: stronger models are not just about bigger code. They are about better representations.

In [ ]:
print("Comparing the classical and neural approaches...")

comparison_df = pd.DataFrame(
    [
        {
            "model": "Classical HOG + Logistic Regression",
            "accuracy": classical_test_metrics["accuracy"],
            "precision": classical_test_metrics["precision"],
            "recall": classical_test_metrics["recall"],
            "f1": classical_test_metrics["f1"],
        },
        {
            "model": "Neural CNN",
            "accuracy": neural_test_metrics["accuracy"],
            "precision": neural_test_metrics["precision"],
            "recall": neural_test_metrics["recall"],
            "f1": neural_test_metrics["f1"],
        },
    ]
).round(3)

display(comparison_df)

comparison_long = comparison_df.melt(id_vars="model", var_name="metric", value_name="score")
fig_compare = px.bar(
    comparison_long,
    x="metric",
    y="score",
    color="model",
    barmode="group",
    text="score",
    title="Test-set comparison: classical baseline vs neural model",
    template="plotly_white",
)
fig_compare.update_traces(texttemplate="%{text:.3f}", textposition="outside")
fig_compare.update_layout(yaxis=dict(range=[0, 1]))
fig_compare.show()

test_results_df = meta_df.iloc[test_idx].copy()
test_results_df["models_disagree"] = test_results_df["classical_pred_id"] != test_results_df["neural_pred_id"]
test_results_df["either_wrong"] = (
    (test_results_df["classical_pred_id"] != test_results_df["label_id"]) |
    (test_results_df["neural_pred_id"] != test_results_df["label_id"])
)

interesting_examples = test_results_df.loc[
    test_results_df["models_disagree"] | test_results_df["either_wrong"],
    [
        "sample_id",
        "label_name",
        "classical_pred",
        "classical_confidence",
        "neural_pred",
        "neural_confidence",
    ],
].head(8)

print(f"Test examples where the models disagree: {int(test_results_df['models_disagree'].sum())}")
display(interesting_examples)

disagreement_indices = test_results_df.loc[test_results_df["models_disagree"], :].index.tolist()[:6]
if len(disagreement_indices) > 0:
    fig, axes = plt.subplots(1, len(disagreement_indices), figsize=(2.6 * len(disagreement_indices), 3))
    if len(disagreement_indices) == 1:
        axes = [axes]
    for ax, idx in zip(axes, disagreement_indices):
        ax.imshow(images[idx], cmap="gray")
        ax.set_title(
            f"True: {meta_df.loc[idx, 'label_name']}\nC: {meta_df.loc[idx, 'classical_pred']}\nN: {meta_df.loc[idx, 'neural_pred']}",
            fontsize=8,
        )
        ax.axis("off")
    plt.suptitle("Examples where the models disagree", fontsize=13)
    plt.tight_layout()
    plt.show()
else:
    print("No disagreements to visualise for this run.")

## 8. Interactive Vision Explorer

<div style="border:1px solid #ffeeba; background:#fff3cd; color:#856404; padding:16px; margin:8px 0; border-radius:4px;">
    <b>Try it yourself:</b> Use the explorer below to choose a test image, apply safe transformations, and compare how the classical and neural models respond.
</div>

This is a powerful way to learn because you can connect:
- the original image
- the transformed image
- the HOG-style representation
- and the two models' predicted probabilities

It is also a gentle introduction to model robustness: small visual changes can affect predictions.

In [ ]:
print("Building the interactive vision explorer...")

test_sample_options = [
    (f"Test sample {i} | true label: {meta_df.loc[idx, 'label_name']}", int(idx))
    for i, idx in enumerate(test_idx[: min(200, len(test_idx))])
]

sample_dropdown = widgets.Dropdown(options=test_sample_options, description="Image:")
rotation_slider = widgets.IntSlider(value=0, min=-25, max=25, step=5, description="Rotate:", continuous_update=False)
blur_slider = widgets.IntSlider(value=0, min=0, max=3, step=1, description="Blur:", continuous_update=False)
run_button = widgets.Button(description="Update explorer", button_style="success", icon="eye")
explorer_output = widgets.Output()

def predict_classical_on_image(image):
    features = hog_features(image).reshape(1, -1)
    probs = classical_model.predict_proba(features)[0]
    pred_id = int(np.argmax(probs))
    return pred_id, probs

def predict_neural_on_image(image):
    image_tensor = torch.tensor(image / 255.0, dtype=torch.float32).unsqueeze(0).unsqueeze(0).to(device)
    neural_model.eval()
    with torch.no_grad():
        probs = torch.softmax(neural_model(image_tensor), dim=1)[0].cpu().numpy()
    pred_id = int(np.argmax(probs))
    return pred_id, probs

def run_explorer(_=None):
    idx = sample_dropdown.value
    rotation = rotation_slider.value
    blur_strength = blur_slider.value
    original = ensure_uint8(images[idx])
    transformed = apply_safe_transform(original, rotation_degrees=rotation, blur_strength=blur_strength)
    _, hog_vis = hog_features(transformed, return_visual=True)

    classical_pred_id, classical_probs = predict_classical_on_image(transformed)
    neural_pred_id, neural_probs = predict_neural_on_image(transformed)

    info_df = pd.DataFrame(
        [
            {
                "sample_id": int(idx),
                "true_label": meta_df.loc[idx, 'label_name'],
                "classical_prediction": f"{class_names[classical_pred_id]} ({np.max(classical_probs):.3f})",
                "neural_prediction": f"{class_names[neural_pred_id]} ({np.max(neural_probs):.3f})",
                "rotation": rotation,
                "blur_strength": blur_strength,
            }
        ]
    )

    prob_df = pd.DataFrame(
        {
            "class_name": class_names + class_names,
            "probability": list(classical_probs) + list(neural_probs),
            "model": ["Classical baseline"] * len(class_names) + ["Neural CNN"] * len(class_names),
        }
    )

    with explorer_output:
        clear_output(wait=True)
        print(f"Exploring sample {idx} from the test set.")
        display(info_df)

        fig, axes = plt.subplots(1, 3, figsize=(10, 3.5))
        axes[0].imshow(original, cmap="gray")
        axes[0].set_title("Original")
        axes[1].imshow(transformed, cmap="gray")
        axes[1].set_title("Transformed")
        axes[2].imshow(hog_vis, cmap="inferno")
        axes[2].set_title("HOG view")
        for ax in axes:
            ax.axis("off")
        plt.tight_layout()
        plt.show()

        fig_probs = px.bar(
            prob_df,
            x="class_name",
            y="probability",
            color="model",
            barmode="group",
            text="probability",
            title="Predicted class probabilities after the chosen transformation",
            template="plotly_white",
        )
        fig_probs.update_traces(texttemplate="%{text:.3f}", textposition="outside")
        fig_probs.update_layout(yaxis=dict(range=[0, 1]), xaxis_title="Class", yaxis_title="Probability")
        fig_probs.show()

run_button.on_click(run_explorer)

display(widgets.VBox([sample_dropdown, rotation_slider, blur_slider, run_button, explorer_output]))
run_explorer()
print("Interactive explorer ready! Try rotating or blurring an image to see how the predictions change.")

## 9. Limitations, Edge Cases, and Responsible Use

<div style="border:1px solid #f5c6cb; background:#f8d7da; color:#721c24; padding:16px; margin:8px 0; border-radius:4px;">
    <b>Be careful:</b> Good notebook results do not mean a vision system will automatically work well in the real world. Image models can be sensitive to lighting, rotation, blur, cropping, resolution, and the kinds of examples included in the training data.
</div>

A few important limitations to remember:

- **Small teaching dataset:** This notebook is designed for learning, not for making state-of-the-art claims.
- **Controlled images:** Classroom datasets are usually cleaner than real deployment settings.
- **Transform sensitivity:** A model may change its prediction when an image is rotated, blurred, or partly obscured.
- **Confidence is not certainty:** High probability does not guarantee correctness.
- **Bias and coverage:** Real computer vision systems can behave unevenly when the training data does not represent the full range of real-world examples.

Responsible vision practice usually includes:
- testing on realistic unseen data
- checking failure cases by hand
- documenting limitations clearly
- avoiding overclaiming what the model understands

## Take-away

<div style="border:1px solid #b8daff; background:#d9edf7; color:#0c5460; padding:16px; margin:8px 0; border-radius:4px;">
    Congratulations! You have built an end-to-end vision intelligence workflow. You explored image data, applied preprocessing with OpenCV and scikit-image, trained a classical HOG-based classifier, trained a neural CNN, compared their performance carefully, and investigated predictions through an interactive explorer.
</div>

### Suggested next steps
- Try changing <code>MAX_SAMPLES</code>, <code>EPOCHS</code>, or the transformation settings in the explorer
- Inspect more mistaken predictions to see which classes are visually similar
- Compare HOG with other feature designs
- Replace the small CNN with a deeper architecture
- Extend the workflow to colour images and multi-channel inputs


![Noteable license](https://raw.githubusercontent.com/YusufNik/Noteable/refs/heads/main/Images/Noteable%20Notebook%20Footer.png)